# IA para telemetría de un carro RC
**Prototipo reproducible — Python + Google Colab**

Este cuaderno implementa un clasificador explicable (árbol de decisión) y un generador de informe en español. Los datos incluidos son **sintéticos**: sirven para comprobar el flujo, no demuestran todavía el rendimiento en un carro físico.

## Qué hace
1. Carga corridas CSV y valida columnas, tiempos y datos faltantes.
2. Divide cada corrida en ventanas de 0,5 s con paso de 0,1 s.
3. Extrae características físicas interpretables.
4. Entrena un árbol pequeño, separando corridas completas.
5. Compara árbol y reglas sobre las mismas ventanas de prueba.
6. Agrupa eventos vecinos y genera texto + JSON por corrida.

**Importante:** se asume aceleración en m/s² y ángulos en grados. `az` contiene la gravedad cuando el sensor está quieto (aprox. 9,81 m/s²). Confirma orientación y unidades reales del MPU6050 antes de usar el modelo en el vehículo.


**Versión 0.2 (revisión del 12 de noviembre de 2026).** Cambios respecto a la versión anterior:
- El impacto se reconoce por la aceleración total, en cualquier dirección (antes solo en el eje X).
- Las inclinaciones que se recuperan (20° a 85°) se entrenan como `normal`: ya no se confunden con vuelco. La inclinación sostenida se informa aparte con una regla explícita.
- «Frenada brusca» tiene un umbral declarado (`BRAKE_THRESHOLD`); la parada suave del final ya no se informa como frenada.
- Las ventanas se adaptan a la frecuencia de muestreo real de cada corrida.
- El momento de cada evento es el instante medido (pico o cruce), no el borde de la ventana.
- Adaptadores para leer sesiones de la API y el Excel de la web; cada evento incluye `api_tipo` (`golpe`, `vuelco`, `inclinacion`).
- El modelo se guarda con versión en `rc_demo_data/modelo_rc.joblib`.

Convención de ángulos (igual que la web y el firmware): `roll_deg` positivo levanta el lado izquierdo, `pitch_deg` positivo levanta la trompa.


## 1. Carga de librerías y configuración

En Colab, las librerías habituales ya están instaladas. Si ejecutas localmente, instala `numpy pandas scikit-learn matplotlib`.

In [ ]:
import os, glob, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

SEED = 42
np.random.seed(SEED)
G = 9.81
MODEL_VERSION = "arbol-rc-0.2"
CLASSES = ["normal", "frenada", "impacto", "vuelco"]
FEATURES = ["acc_dyn_peak", "acc_norm_delta_peak", "longitudinal_min_smooth",
            "tilt_peak_deg", "tilt_time_s"]
REQUIRED = ["run_id", "origin", "t_ms", "ax", "ay", "az", "roll_deg", "pitch_deg"]

# Umbrales del informe: provisionales, se recalibran con corridas físicas.
BRAKE_THRESHOLD = -4.0      # m/s², mínimo de ax suavizado (0,1 s) para llamar "frenada brusca"
ROLLOVER_DEG = 90.0         # inclinación a partir de la cual cuenta el tiempo de vuelco
TILT_WARN_DEG = 60.0        # mismo valor inicial que el formulario de la web
TILT_WARN_S = 0.10          # permanencia mínima de la inclinación


## 2. Generación de datos sintéticos de demostración

Cada corrida es independiente y tiene tiempos e intensidades distintos. La gravedad se proyecta en los ejes según la postura, con el mismo modelo que usa la web.

- `normal`: avance, vibración y parada suave al final.
- `frenada`: desaceleración longitudinal fuerte, sin golpe. Solo se etiqueta el tramo que supera `BRAKE_THRESHOLD`.
- `impacto`: pico de 1 a 5 muestras en una dirección aleatoria del plano (frontal, trasero, lateral u oblicuo).
- `vuelco`: la postura pasa de 90° y se queda ahí. La mitad incluye un golpe al inicio.
- `inclinacion`: se inclina entre 20° y 85° y se recupera. Se etiqueta `normal`.


In [ ]:
def make_run(run_id, scenario, seed, origin="simulado", fs=50, tilt_deg=45.0):
    rng = np.random.default_rng(seed)
    duration = 10.0
    t = np.arange(0, duration, 1/fs)
    n = len(t)
    ax = rng.normal(0, 0.35, n)
    ax += rng.uniform(0, 3.3)*np.sin(np.pi*np.clip((t-.5)/1.,0,1))
    ay = rng.normal(0, 0.25, n)
    az = G + rng.normal(0, 0.25, n)
    if scenario in {"normal", "inclinacion"}:
        # Parada suave al final: NO es frenada brusca.
        ax -= rng.uniform(2.2, 3.0)*np.sin(np.pi*np.clip((t-8)/1.,0,1))
    roll = rng.normal(0, 1.2, n)
    pitch = rng.normal(0, 1.2, n)

    event_t = float(rng.uniform(3.0, 7.0))
    event_duration = float(rng.uniform(0.35, 0.9))
    impact_slice = None

    if scenario == "frenada":
        mask = (t >= event_t) & (t <= event_t + event_duration)
        brake = np.zeros(n)
        brake[mask] = rng.uniform(5.0, 11.0) * np.sin(np.pi*(t[mask]-event_t)/event_duration)
        ax -= brake
    elif scenario == "impacto":
        idx = int(np.argmin(np.abs(t-event_t)))
        width = int(rng.integers(0, 3))            # de 1 a 5 muestras
        amp = rng.uniform(14, 32)
        # Dirección aleatoria en el plano: frontal, trasero, lateral u oblicuo.
        ang = rng.uniform(0, 2*np.pi)
        sl = slice(max(0, idx-width), min(n, idx+width+1))
        ax[sl] += amp*np.cos(ang); ay[sl] += amp*np.sin(ang)
        az[sl] += rng.normal(0, 2.0)
        if rng.random() < 0.35:
            roll += np.exp(-((t-event_t)/0.22)**2) * rng.uniform(8, 22)
        impact_slice = sl
    elif scenario == "vuelco":
        ramp = np.clip((t-event_t)/rng.uniform(0.35, 1.2), 0, 1)
        final_angle = rng.choice([-1, 1]) * rng.uniform(125, 178)
        roll = final_angle*ramp + rng.normal(0, 2, n)
        if seed % 2 == 0 or run_id.startswith("DEMO"):
            idx = int(np.argmin(np.abs(t-event_t)))
            ang = rng.uniform(0, 2*np.pi); amp = rng.uniform(16, 25)
            impact_slice = slice(max(0, idx-1), min(n, idx+2))
            ax[impact_slice] += amp*np.cos(ang); ay[impact_slice] += amp*np.sin(ang)
    elif scenario == "inclinacion":
        # Se inclina tilt_deg y se recupera: nunca es vuelco, por grande que sea el ángulo (< 90°).
        roll += rng.choice([-1, 1]) * tilt_deg * np.exp(-((t-(event_t+0.35))/rng.uniform(0.35, 0.8))**2)
    elif scenario != "normal":
        raise ValueError("Escenario no reconocido")

    # La gravedad se proyecta según la postura en TODOS los escenarios (mismo modelo que la web).
    r = np.deg2rad(roll); p = np.deg2rad(pitch)
    ax = ax + G*np.sin(p)
    ay = ay + G*np.sin(r)*np.cos(p)
    az = az - G + G*np.cos(r)*np.cos(p)

    df = pd.DataFrame({"run_id": run_id, "origin": origin, "t_ms": np.round(t*1000).astype(int),
                       "ax": ax, "ay": ay, "az": az, "roll_deg": roll, "pitch_deg": pitch})
    df["scenario_reference"] = scenario
    label = np.full(n, "normal", dtype=object)
    if scenario == "frenada":
        # Frenada brusca = tramo donde la desaceleración aplicada supera el umbral declarado.
        label[brake >= abs(BRAKE_THRESHOLD)] = "frenada"
    elif scenario == "vuelco":
        # Vuelco = postura más allá de 90°. La rampa previa todavía no lo es.
        label[np.abs(roll) > ROLLOVER_DEG] = "vuelco"
    if impact_slice is not None:
        label[impact_slice] = "impacto"          # exactamente las muestras del golpe
    df["label"] = label
    return df

# Dataset: variantes aleatorias por corrida. Las inclinaciones recuperadas cubren de 20° a 85°.
scenario_counts = {"normal": 14, "frenada": 14, "impacto": 20, "vuelco": 14, "inclinacion": 16}
runs = []
seed_counter = 100
tilt_rng = np.random.default_rng(SEED)
for sc, count in scenario_counts.items():
    for j in range(count):
        runs.append(make_run(f"SIM_{sc.upper()}_{j+1:02d}", sc, seed_counter,
                             tilt_deg=float(tilt_rng.uniform(20, 85))))
        seed_counter += 1
all_data = pd.concat(runs, ignore_index=True)
print("Corridas:", all_data.run_id.nunique(), "| muestras:", len(all_data))
print(all_data.groupby("scenario_reference").run_id.nunique())


## 3. Carga y revisión de corridas

Formato mínimo: `run_id, origin, t_ms, ax, ay, az, roll_deg, pitch_deg`. Para entrenar se necesita además `label` por muestra. `scenario_reference` solo describe la demostración y **nunca entra al modelo**.

Para datos de la web hay dos adaptadores:
- `df_from_api(sesion, mediciones)`: recibe lo que devuelve la API (`sesion.id`, `sesion.origen` y la lista de mediciones).
- `df_from_web_excel(ruta)`: lee el `.xlsx` exportado desde la web.


In [ ]:
def validate_data(df):
    missing_cols = [c for c in REQUIRED if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Faltan columnas obligatorias: {missing_cols}")
    out = df.copy()
    numeric = ["t_ms", "ax", "ay", "az", "roll_deg", "pitch_deg"]
    for c in numeric:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    out = out.sort_values(["run_id", "t_ms"]).reset_index(drop=True)
    missing_by_run = out.groupby("run_id")[numeric].apply(lambda x: int(x.isna().sum().sum()))
    if out[numeric].isna().any().any():
        warnings.warn("Hay datos faltantes; se mantienen para reportarlos y se excluyen ventanas incompletas.")
    return out, missing_by_run

# --- Adaptadores entre el contrato de la web/API y el cuaderno -------------------------------
API_EVENT_TYPE = {"impacto": "golpe", "vuelco": "vuelco", "inclinacion": "inclinacion"}  # 'frenada' no es evento de la API

def df_from_api(sesion, mediciones):
    """sesion = {"id","origen"}; mediciones = lista del endpoint /sesiones/:id/mediciones."""
    df = pd.DataFrame(mediciones)
    df["run_id"] = sesion["id"]
    df["origin"] = sesion["origen"]
    return df[REQUIRED + [c for c in ["numero", "rueda_rps"] if c in df.columns]]

def df_from_web_excel(path):
    """Lee el .xlsx exportado por la web (hojas Datos y Metadatos)."""
    datos = pd.read_excel(path, sheet_name="Datos")
    meta = pd.read_excel(path, sheet_name="Metadatos")
    if {"campo", "valor"} <= set(meta.columns):          # Excel generado por el servidor
        m = dict(zip(meta.campo, meta.valor)); sesion = {"id": m["sesion_id"], "origen": m["origen"]}
    else:                                                # Excel generado en el navegador
        sesion = {"id": meta.sesion.iloc[0], "origen": meta.origen.iloc[0]}
    return df_from_api(sesion, datos.to_dict("records"))

checked, missing_report = validate_data(all_data)
print("Validación completada. Corridas con datos faltantes:", int((missing_report > 0).sum()))


## 4. Fragmentación y extracción de características

Ventanas de **0,5 s** con paso de **0,1 s**. El número mínimo de muestras por ventana se calcula con el periodo de muestreo real de la corrida.

Características:
- `acc_dyn_peak`: máximo de |A − 9,81|, con A = √(ax²+ay²+az²). Mide cuánto se aparta la señal de la gravedad, en cualquier dirección.
- `acc_norm_delta_peak`: mayor cambio de A entre muestras consecutivas.
- `longitudinal_min_smooth`: mínimo de `ax` promediado en 0,1 s. Ignora picos de una sola muestra.
- `tilt_peak_deg`: mayor |roll| o |pitch|.
- `tilt_time_s`: segundos de la ventana con inclinación mayor a 90°.

Etiqueta de la ventana: `impacto` si contiene alguna muestra de golpe; `vuelco` si acumula 0,25 s más allá de 90°; `frenada` si al menos el 20 % de la ventana es frenada brusca; `normal` en otro caso.


In [ ]:
WINDOW_S, STEP_S, SMOOTH_S = 0.5, 0.1, 0.1

def sample_period(t_ms):
    dt = np.diff(np.asarray(t_ms, float))/1000
    dt = dt[dt > 0]
    return float(np.median(dt)) if len(dt) else 0.02

def extract_features(window, period):
    ax = window.ax.to_numpy(float); ay = window.ay.to_numpy(float); az = window.az.to_numpy(float)
    an = np.sqrt(ax**2 + ay**2 + az**2)
    d_an = np.abs(np.diff(an))
    tilt = np.maximum(np.abs(window.roll_deg.to_numpy(float)), np.abs(window.pitch_deg.to_numpy(float)))
    k = max(1, int(round(SMOOTH_S/period)))               # suavizado de 0,1 s, vale para cualquier muestreo
    ax_smooth = np.convolve(ax, np.ones(k)/k, mode="valid") if len(ax) >= k else ax
    return {
        "acc_dyn_peak": float(np.max(np.abs(an - G))),      # cuánto se aparta de la gravedad, en cualquier dirección
        "acc_norm_delta_peak": float(np.max(d_an)) if len(d_an) else 0.0,
        "longitudinal_min_smooth": float(np.min(ax_smooth)),
        "tilt_peak_deg": float(np.max(tilt)),
        "tilt_time_s": float(np.sum(tilt > ROLLOVER_DEG) * period),
    }

def make_windows(df, labeled=True):
    records = []
    sensor_cols = ["ax", "ay", "az", "roll_deg", "pitch_deg"]
    for run_id, g in df.groupby("run_id"):
        g = g.sort_values("t_ms").reset_index(drop=True)
        t = g.t_ms.to_numpy(float)/1000
        if len(t) < 2: continue
        period = sample_period(g.t_ms)
        min_samples = max(5, int(0.8*WINDOW_S/period))      # depende del muestreo real, no de 50 Hz fijos
        s = t[0]
        while s + WINDOW_S <= t[-1] + period + 1e-9:
            w = g[(t >= s - 1e-9) & (t < s + WINDOW_S - 1e-9)]
            if len(w) >= min_samples and not w[sensor_cols].isna().any().any():
                rec = {"run_id": run_id, "origin": g.origin.iloc[0], "t_start_s": float(s),
                       "t_end_s": float(s+WINDOW_S), "t_center_s": float(s+WINDOW_S/2)}
                rec.update(extract_features(w, period))
                if labeled and "label" in g:
                    counts = w.label.value_counts()
                    # Un evento breve no debe desaparecer por ser minoritario en la ventana.
                    if "impacto" in counts.index: best = "impacto"
                    elif counts.get("vuelco", 0)*period >= 0.25: best = "vuelco"
                    elif counts.get("frenada", 0) >= max(3, int(0.20*len(w))): best = "frenada"
                    else: best = "normal"
                    rec["y"] = best
                records.append(rec)
            s += STEP_S
    return pd.DataFrame(records)

windows = make_windows(checked)
print("Ventanas:", len(windows), "| corridas:", windows.run_id.nunique())
print(windows.y.value_counts())


## 5. Separación por corridas completas y entrenamiento

La división se hace por `run_id`: ninguna ventana de una corrida puede aparecer a la vez en entrenamiento y prueba. La semilla fija permite repetir el reparto. Reservamos corridas completas de cada escenario para prueba final. Las corridas de inclinación recuperada también se reservan parcialmente para medir falsas alarmas.

In [ ]:
# Reparto determinista por corrida y escenario de referencia; el escenario solo organiza el split.
run_meta = all_data.groupby("run_id").agg(scenario=("scenario_reference","first"), origin=("origin","first")).reset_index()
train_ids, test_ids = [], []
rng = np.random.default_rng(SEED)
for scenario, grp in run_meta.groupby("scenario"):
    ids = grp.run_id.to_numpy().copy()
    rng.shuffle(ids)
    n_test = max(2, int(round(0.3*len(ids))))
    test_ids.extend(ids[:n_test]); train_ids.extend(ids[n_test:])

train_w = windows[windows.run_id.isin(train_ids)].copy()
test_w = windows[windows.run_id.isin(test_ids)].copy()
assert set(train_w.run_id).isdisjoint(set(test_w.run_id))
X_train, y_train = train_w[FEATURES], train_w["y"]
X_test, y_test = test_w[FEATURES], test_w["y"]
print("Corridas entrenamiento:", len(train_ids), "| prueba:", len(test_ids))

model = DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, class_weight="balanced", random_state=SEED)
model.fit(X_train, y_train)
print("\nReglas aprendidas por el árbol:")
print(export_text(model, feature_names=FEATURES))


## 6. Comparación con detector por reglas

Reglas de referencia, explícitas y modificables:
- vuelco: más de 0,25 s de la ventana por encima de 90°;
- impacto: aceleración total que se aparta más de 10 m/s² de la gravedad, o cambio entre muestras mayor a 12 m/s²;
- frenada: `ax` suavizado por debajo de `BRAKE_THRESHOLD`;
- normal: lo demás.

Son umbrales para datos sintéticos. Árbol y reglas se evalúan sobre las mismas ventanas de las corridas reservadas.


In [ ]:
def rule_predict(row):
    if row.tilt_time_s > 0.25:
        return "vuelco"
    if row.acc_dyn_peak > 10 or row.acc_norm_delta_peak > 12:
        return "impacto"
    if row.longitudinal_min_smooth <= BRAKE_THRESHOLD:
        return "frenada"
    return "normal"

test_w = test_w.copy()
test_w["tree_pred"] = model.predict(X_test)
test_w["rule_pred"] = test_w.apply(rule_predict, axis=1)

def show_metrics(y_true, y_pred, title):
    print("\n"+title)
    print("Exactitud:", round(accuracy_score(y_true,y_pred),4))
    print(classification_report(y_true,y_pred,labels=CLASSES,zero_division=0))
    cm=confusion_matrix(y_true,y_pred,labels=CLASSES)
    display(pd.DataFrame(cm,index=[f"real_{c}" for c in CLASSES], columns=[f"pred_{c}" for c in CLASSES]))

show_metrics(test_w.y, test_w.tree_pred, "ÁRBOL — ventanas de corridas de prueba")
show_metrics(test_w.y, test_w.rule_pred, "REGLAS — mismas ventanas de prueba")

tilt_ids=set(run_meta.loc[run_meta.scenario=="inclinacion","run_id"]) & set(test_ids)
tilt_eval=test_w[test_w.run_id.isin(tilt_ids)]
print("Ventanas de inclinación recuperada en prueba:", len(tilt_eval))
print("Falsas alarmas de vuelco — árbol:", int((tilt_eval.tree_pred=="vuelco").sum()))
print("Falsas alarmas de vuelco — reglas:", int((tilt_eval.rule_pred=="vuelco").sum()))


## 7. Tabla por corrida: referencia, árbol y reglas

La referencia se obtiene de las etiquetas de muestra del simulador, no del nombre del archivo. Una corrida puede contener más de un evento; por eso se muestran las clases detectadas sin eliminar el orden temporal.

In [ ]:
def summarize_run_predictions(wdf, pred_col):
    return {rid: ", ".join(c for c in CLASSES if c!="normal" and (g[pred_col]==c).any()) or "normal"
            for rid,g in wdf.groupby("run_id")}

tree_by_run=summarize_run_predictions(test_w,"tree_pred")
rules_by_run=summarize_run_predictions(test_w,"rule_pred")
rows=[]
for rid in test_ids:
    g=all_data[all_data.run_id==rid]
    actual=[c for c in ["frenada","impacto","vuelco"] if (g.label==c).any()] or ["normal"]
    rows.append({"corrida":rid,"referencia":", ".join(actual),
                 "árbol (ventanas)":tree_by_run.get(rid,"sin ventanas"),
                 "reglas":rules_by_run.get(rid,"sin ventanas")})
run_comparison=pd.DataFrame(rows)
display(run_comparison)


## 8. Generador de informe final en español + JSON

Las ventanas consecutivas de la misma clase se fusionan en un evento. Para cada evento se informa:
- `time_s`: instante medido (pico del golpe, primer cruce de 90° o máxima desaceleración);
- `confidence`: proporción de la hoja del árbol. **No es una probabilidad calibrada**;
- `source`: `arbol` o `regla`;
- `api_tipo`: nombre del evento en la API (`golpe`, `vuelco`, `inclinacion`). La frenada no es evento de la API y queda en `null`.

La inclinación sostenida (≥ `TILT_WARN_DEG` durante `TILT_WARN_S`, sin llegar a vuelco) se detecta con una regla, igual que en la web.

El veredicto pasa a `incierto` si falta más del 5 % de las lecturas, si hay un corte de señal o si algún evento tiene `confidence` menor a 0,45.


In [ ]:
def merge_events(pred_windows, pred_col="prediction", gap_s=0.25):
    events = []
    p = pred_windows.sort_values("t_center_s")
    for _, r in p[p[pred_col] != "normal"].iterrows():
        cls = r[pred_col]
        if events and events[-1]["type"] == cls and r.t_start_s <= events[-1]["end_s"] + gap_s:
            events[-1]["end_s"] = max(events[-1]["end_s"], float(r.t_end_s))
            events[-1]["_probs"].append(float(r.max_probability))
        else:
            events.append({"type": cls, "start_s": float(r.t_start_s), "end_s": float(r.t_end_s),
                           "_probs": [float(r.max_probability)]})
    return events

def make_verdict(run_df, fitted_model=None):
    fitted_model = fitted_model or model
    g, _ = validate_data(run_df)
    rid = str(g.run_id.iloc[0]); origin = str(g.origin.iloc[0])
    numeric = ["t_ms", "ax", "ay", "az", "roll_deg", "pitch_deg"]
    total = len(g)
    miss_count = int(g[numeric].isna().sum().sum())
    g = g.dropna(subset=numeric).reset_index(drop=True)
    t = g.t_ms.to_numpy(float)/1000
    period = sample_period(g.t_ms)
    gaps = np.diff(t)
    gap_flag = bool(len(gaps) and np.any(gaps > max(0.1, 3*period)))
    backwards = bool(len(gaps) and np.any(gaps < 0))
    w = make_windows(g, labeled=False) if not backwards else pd.DataFrame()
    base = {"run_id": rid, "origin": origin, "model_version": MODEL_VERSION}
    if len(w) == 0:
        return {**base, "status": "incierto", "events": [], "final_posture": "no estimable",
                "text": "Datos insuficientes para determinar qué ocurrió."}
    probs = fitted_model.predict_proba(w[FEATURES])
    w["prediction"] = fitted_model.predict(w[FEATURES])
    w["max_probability"] = probs.max(axis=1)
    # Filtro declarado: solo es "frenada brusca" si la desaceleración suavizada supera el umbral.
    w.loc[(w.prediction == "frenada") & (w.longitudinal_min_smooth > BRAKE_THRESHOLD), "prediction"] = "normal"
    events = merge_events(w)

    an = np.sqrt(g.ax**2 + g.ay**2 + g.az**2).to_numpy(float)
    tilt = np.maximum(g.roll_deg.abs(), g.pitch_deg.abs()).to_numpy(float)
    k = max(1, int(round(SMOOTH_S/period)))
    ax_s = pd.Series(g.ax.to_numpy(float)).rolling(k, center=True, min_periods=1).mean().to_numpy()

    # Momento de cada evento = instante medido dentro de su intervalo, no el borde de la ventana.
    for e in events:
        m = (t >= e["start_s"] - 1e-9) & (t <= e["end_s"] + 1e-9)
        idx = np.flatnonzero(m)
        if e["type"] == "impacto":
            i = idx[np.argmax(np.abs(an[m] - G))]
        elif e["type"] == "vuelco":
            over = idx[tilt[m] > ROLLOVER_DEG]; i = over[0] if len(over) else idx[np.argmax(tilt[m])]
        else:
            i = idx[np.argmin(ax_s[m])]
        e["time_s"] = round(float(t[i]), 2)
        e["start_s"] = round(e["start_s"], 2); e["end_s"] = round(e["end_s"], 2)
        e["peak_acceleration_m_s2"] = round(float(an[m].max()), 2)
        e["min_longitudinal_m_s2"] = round(float(ax_s[m].min()), 2)
        e["max_tilt_deg"] = round(float(tilt[m].max()), 2)
        e["confidence"] = round(float(np.mean(e.pop("_probs"))), 2)
        e["source"] = "arbol"

    # Inclinación sostenida sin vuelco: regla explícita (misma idea que la web), no sale del árbol.
    if not any(e["type"] == "vuelco" for e in events):
        run_len, start_i = 0, None
        for i, v in enumerate((tilt >= TILT_WARN_DEG) & (tilt <= ROLLOVER_DEG)):
            if v:
                start_i = i if run_len == 0 else start_i; run_len += 1
                if run_len*period >= TILT_WARN_S:
                    j = i
                    while j+1 < len(tilt) and tilt[j+1] >= TILT_WARN_DEG: j += 1
                    seg = slice(start_i, j+1)
                    events.append({"type": "inclinacion", "start_s": round(float(t[start_i]), 2), "end_s": round(float(t[j]), 2),
                                   "time_s": round(float(t[start_i]), 2), "peak_acceleration_m_s2": round(float(an[seg].max()), 2),
                                   "min_longitudinal_m_s2": round(float(ax_s[seg].min()), 2),
                                   "max_tilt_deg": round(float(tilt[seg].max()), 2), "confidence": None, "source": "regla"})
                    break
            else:
                run_len = 0
    events.sort(key=lambda e: e["time_s"])
    for e in events:
        e["api_tipo"] = API_EVENT_TYPE.get(e["type"])      # None para 'frenada': la API no la registra como evento

    low_conf = any(e["confidence"] is not None and e["confidence"] < 0.45 for e in events)
    uncertain = (miss_count > 0.05*total*6) or gap_flag or low_conf
    final_roll = float(g.roll_deg.iloc[-1]); final_pitch = float(g.pitch_deg.iloc[-1])
    final_tilt = max(abs(final_roll), abs(final_pitch))
    posture = ("volcado al final" if final_tilt > ROLLOVER_DEG else
               "inclinado al final" if final_tilt > 25 else "aproximadamente nivelado")

    status = "incierto" if uncertain else ("eventos_detectados" if events else "normal")
    if uncertain:
        sentence = "Datos insuficientes para determinar qué ocurrió con confianza; se requiere revisar la señal y/o el video."
    elif not events:
        sentence = "Durante esta corrida no se detectaron señales de frenada brusca, impacto, inclinación sostenida ni vuelco."
    else:
        parts = []
        for e in events:
            phrase = f"{e['type'].replace('inclinacion','inclinación sostenida')} cerca del segundo {e['time_s']:.2f}"
            if e["type"] == "impacto": phrase += f", con pico de aceleración total de {e['peak_acceleration_m_s2']:.2f} m/s²"
            if e["type"] == "frenada": phrase += f", con desaceleración de {abs(e['min_longitudinal_m_s2']):.2f} m/s²"
            if e["type"] in ("vuelco", "inclinacion"): phrase += f", con inclinación máxima de {e['max_tilt_deg']:.1f}°"
            parts.append(phrase)
        sentence = "Se detectó " + "; seguido de ".join(parts) + "."
    if gap_flag:
        sentence += " Se observó un corte o salto temporal de señal; revisar el intervalo."
    tilt_peak = float(tilt.max())
    if 35 <= tilt_peak <= ROLLOVER_DEG and not any(e["type"] in ("inclinacion", "vuelco") for e in events):
        sentence += f" La inclinación máxima fue {tilt_peak:.1f}°, sin llegar a inclinación sostenida ni vuelco."
    sentence += f" El carro terminó {posture}."
    return {**base, "status": status, "events": events,
            "peak_acceleration_m_s2": round(float(an.max()), 2),
            "min_longitudinal_m_s2": round(float(ax_s.min()), 2),
            "max_tilt_deg": round(tilt_peak, 2),
            "final_roll_deg": round(final_roll, 2), "final_pitch_deg": round(final_pitch, 2),
            "final_posture": posture, "duration_s": round(float(t[-1]-t[0]), 2),
            "sample_rate_hz": round(1/period, 1),
            "missing_numeric_values": miss_count, "signal_gap_detected": gap_flag,
            "uncertainty_note": "confidence es la proporción de la hoja del árbol, no una probabilidad calibrada.",
            "text": sentence}


## 9. Cinco corridas demostrativas

Se generan corridas independientes para los cinco casos. Las predicciones se muestran tal como salen del modelo; no se corrigen para coincidir con la referencia.


In [ ]:
demo_scenarios=["normal","frenada","impacto","vuelco","inclinacion"]
demo_runs={}
for i,sc in enumerate(demo_scenarios):
    d=make_run(f"DEMO_{sc.upper()}",sc,900+i,tilt_deg=45.0)
    demo_runs[sc]=d
    verdict=make_verdict(d)
    print("\n"+"="*72)
    print("REFERENCIA:",sc,"| VEREDICTO:",verdict["status"])
    print(verdict["text"])
    print("Eventos:", [(e["type"], e["time_s"], e["source"]) for e in verdict["events"]])


## 10. Exportación de JSON y guía para incorporar el carro físico

El JSON de salida es el contrato inicial para el botón **«¿Qué pasó en esta corrida?»**. Mantiene `run_id`, `origin`, eventos ordenados, tiempos en segundos y unidades explícitas.

Para datos físicos:
1. Exporta el Excel desde la web y cárgalo con `df_from_web_excel`, o usa un CSV con las ocho columnas obligatorias.
2. Añade `label` solo si vas a entrenar/evaluar con anotaciones revisadas por video.
3. El firmware ya envía m/s² y grados. Configura el MPU6050 en ±8 g o ±16 g: con ±2 g los golpes se recortan.
4. Documenta qué eje apunta hacia delante y cuál hacia arriba. Si cambias orientación, adapta `ax`/`ay`/`az`.
5. No mezcles corridas físicas en entrenamiento y prueba: separa por corrida y reporta resultados físicos aparte.
6. El sensor Hall puede añadirse luego como característica opcional, pero este prototipo no depende de él.

**Limitaciones:** el conjunto sintético no reproduce toda la dinámica, vibración, montaje ni saturación del sensor real. Un árbol entrenado con estos datos es una demostración de flujo, no un detector validado. Ajusta umbrales y reentrena solo con corridas anotadas reales.

In [ ]:
# Guardar CSV de las cinco demostraciones, un JSON de veredictos y el modelo versionado.
os.makedirs("rc_demo_data", exist_ok=True)
demo_json=[]
for sc, d in demo_runs.items():
    path=f"rc_demo_data/{d.run_id.iloc[0]}.csv"
    d.drop(columns=["scenario_reference","label"]).to_csv(path,index=False)
    demo_json.append(make_verdict(d))
with open("rc_demo_data/veredictos_demo.json","w",encoding="utf-8") as f:
    json.dump(demo_json,f,ensure_ascii=False,indent=2)
# El servicio de análisis carga este archivo en lugar de reentrenar.
joblib.dump({"model": model, "features": FEATURES, "version": MODEL_VERSION,
             "window_s": WINDOW_S, "step_s": STEP_S, "brake_threshold": BRAKE_THRESHOLD,
             "rollover_deg": ROLLOVER_DEG, "trained_on": "sintético"}, "rc_demo_data/modelo_rc.joblib")
print("Archivos exportados:")
print("\n".join(sorted(glob.glob("rc_demo_data/*"))))


## 11. Elegir una corrida y obtener el veredicto (última celda)

En Colab puedes cambiar `SELECTED_RUN` por el ID de cualquier corrida cargada. Para una corrida nueva, carga su CSV, asegúrate de que `run_id` y `origin` estén presentes y ejecuta esta celda. El bloque imprime el texto y el JSON completo.

In [ ]:
# Cambia este valor por una corrida de demo o por el ID de una corrida física cargada.
SELECTED_RUN = "DEMO_VUELCO"  # ejemplo: DEMO_NORMAL

selected_df = None
for sc, d in demo_runs.items():
    if d.run_id.iloc[0] == SELECTED_RUN:
        selected_df = d
        break

if selected_df is None:
    # Para CSV subidos en Colab, por ejemplo: /content/mi_corrida.csv
    CSV_PATH = None  # ruta de un .csv o del .xlsx exportado por la web
    if CSV_PATH and os.path.exists(CSV_PATH):
        selected_df = df_from_web_excel(CSV_PATH) if CSV_PATH.endswith(".xlsx") else pd.read_csv(CSV_PATH)
    else:
        raise ValueError("No se encontró SELECTED_RUN. Elige uno de demo_runs o define CSV_PATH.")

final_verdict = make_verdict(selected_df)
print(final_verdict["text"])
print("\nJSON para la web:")
print(json.dumps(final_verdict,ensure_ascii=False,indent=2))
